# PD Business Impact – Book Stats, RoE & Threshold Sensitivity

**Purpose:** Demonstrate business impact of the trained PD model using out-of-time test set: approval/charge-off rates, return on equity (RoE), and threshold sensitivity. Compares **baseline (approve all)** vs **model at optimal threshold**.

**Inputs (from pipeline):** Trained model (`pd_model_local_v2.pkl`), test set from `lendingclub_engineered.parquet` (with `loan_amnt`, `int_rate`, `term_months` for business metrics). Run **01** and **02a** first so the parquet includes `loan_amnt` and `int_rate` (01 saves them for this demo).

---

**Assumption – Cost of funds:** We use **cost_of_funds = 3.0%** as a reasonable assumption for the cost at which the bank funds the loans (e.g. wholesale funding). This is documented here and used in RoE calculations. Change the constant in code if your environment uses a different rate.

## 1. Load trained model and test set

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if not (ROOT / "credit_risk").is_dir():
    raise RuntimeError("Run this notebook from the repo root or notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credit_risk.feature_engineering.common_features import get_feature_names_no_leakage_v2
from credit_risk.feature_engineering.feature_screening import screen_features_train_only

DATA_PATH = ROOT / "data" / "credit_risk_pd" / "LendingClub" / "processed" / "lendingclub_engineered.parquet"
MODEL_PATH = ROOT / "models" / "pd" / "pd_model_local_v2.pkl"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_pd_lendingclub_feature_engineering.ipynb first.")
if not MODEL_PATH.exists():
    raise FileNotFoundError("Run 02a_pd_xgboost_training.ipynb and save the model first.")

df = pd.read_parquet(DATA_PATH)
for col in ["loan_amnt", "int_rate"]:
    if col not in df.columns:
        raise ValueError(f"Parquet missing '{col}'. Re-run 01 to add loan_amnt and int_rate for the demo.")

all_feature_names = get_feature_names_no_leakage_v2()
X = df[[c for c in all_feature_names if c in df.columns]].copy()
y = df["default"]
for c in all_feature_names:
    if c not in X.columns:
        X[c] = 0.0
X = X[all_feature_names]

test_idx = df["split"] == "test"
X_test = X[test_idx]
y_test = y[test_idx]
df_test = df[test_idx].copy()

# Feature screening (same as 02a): use train to get selected features
train_idx = df["split"] == "train"
X_train = X[train_idx]
y_train = y[train_idx]
screening = screen_features_train_only(X_train, y_train, missingness_threshold=0.50, min_ks=0.001, corr_threshold=0.95)
feature_names = screening.selected_features
X_test_filled = X_test[feature_names].copy()
medians = X_train[feature_names].median()
X_test_filled = X_test_filled.fillna(medians)

model_data = joblib.load(MODEL_PATH)
final_model = model_data["model"]
best_threshold_opt = float(model_data.get("best_threshold", 0.5))
p_test = final_model.predict_proba(X_test_filled)[:, 1]

# Business df: loan_amnt, int_rate, term (months)
term_col = "term_months" if "term_months" in df_test.columns else "term"
if term_col not in df_test.columns:
    raise ValueError("Parquet missing term_months/term. Re-run 01.")
loans_df = df_test[["loan_amnt", "int_rate", term_col]].rename(columns={term_col: "term"})
if loans_df["term"].dtype == object:
    loans_df["term"] = pd.to_numeric(loans_df["term"].astype(str).str.replace(r" months?", "", regex=True), errors="coerce")
loans_df = loans_df.astype(float, errors="ignore")

n_test = len(y_test)
print(f"Test set: {n_test:,} loans. Optimal threshold (from model pkl): {best_threshold_opt:.3f}")

## 2. Book stats and RoE helpers

In [ ]:
def book_stats(loans_df, p_prob, threshold, label, y_true):
    """Approved = loans where p_prob < threshold (predicted non-default). CoR = charged-off amount / approved amount."""
    y_true = np.asarray(y_true).ravel()
    approved_mask = (np.asarray(p_prob).ravel() < threshold)
    approved_amnt = loans_df.iloc[approved_mask]["loan_amnt"].sum()
    charged_off_amnt = loans_df.iloc[approved_mask & (y_true == 1)]["loan_amnt"].sum()
    n_approved = approved_mask.sum()
    n_total = len(loans_df)
    approval_rate = n_approved / n_total if n_total else 0
    CoR = charged_off_amnt / approved_amnt if approved_amnt else 0
    app = loans_df.iloc[approved_mask]
    wgt_int = (app["int_rate"] * app["loan_amnt"]).sum()
    wgt_int_rate = wgt_int / approved_amnt if approved_amnt else 0
    wgt_term = (app["term"] * app["loan_amnt"]).sum() / approved_amnt if approved_amnt else 0
    print(f"[{label}] Approval rate: {approval_rate:.2%} | CoR: {CoR:.2%} | Weighted avg int rate: {wgt_int_rate:.2f}% | Weighted avg term (mo): {wgt_term:.1f}")
    return {
        "approved_amnt": approved_amnt,
        "charged_off_amnt": charged_off_amnt,
        "approval_rate": approval_rate,
        "CoR": CoR,
        "wgt_int_rate": wgt_int_rate,
        "wgt_term": wgt_term,
        "n_approved": n_approved,
    }

In [ ]:
def return_on_equity(approved_amnt, charge_off_amnt, wgt_int_rate, wgt_term, cost_of_funds=3.0):
    """RoE = (interest_earned - interest_paid - charge_off) / approved. All $ in millions for print."""
    interest_earned = approved_amnt * (wgt_int_rate / 100) * (wgt_term / 12)
    interest_paid = approved_amnt * (cost_of_funds / 100) * (wgt_term / 12)
    banking_profit = interest_earned - interest_paid - charge_off_amnt
    RoE_pct = (banking_profit / approved_amnt * 100) if approved_amnt else 0
    scale = 1e6
    print(f"  Interest earned ($m): {interest_earned/scale:.2f} | Interest paid ($m): {interest_paid/scale:.2f} | Charge-off ($m): {charge_off_amnt/scale:.2f}")
    print(f"  Banking profit ($m): {banking_profit/scale:.2f} | RoE: {RoE_pct:.2f}%")
    return RoE_pct

## 3. Baseline book stats (approve all – no model)

Baseline = approve every loan (threshold 1.0, so all applications are "approved").

In [ ]:
COST_OF_FUNDS = 3.0  # % (documented assumption)
# Baseline: approve all (threshold=1.0 so p_test < 1.0 for every loan)
baseline_stats = book_stats(loans_df, p_test, threshold=1.0, label="Baseline (approve all)", y_true=y_test.values)
return_on_equity(
    baseline_stats["approved_amnt"],
    baseline_stats["charged_off_amnt"],
    baseline_stats["wgt_int_rate"],
    baseline_stats["wgt_term"],
    cost_of_funds=COST_OF_FUNDS,
)

## 4. Model book stats at optimal threshold

In [ ]:
model_stats = book_stats(loans_df, p_test, threshold=best_threshold_opt, label=f"Model (threshold={best_threshold_opt:.2f})", y_true=y_test.values)
return_on_equity(
    model_stats["approved_amnt"],
    model_stats["charged_off_amnt"],
    model_stats["wgt_int_rate"],
    model_stats["wgt_term"],
    cost_of_funds=COST_OF_FUNDS,
)

## 5. Threshold sensitivity analysis

Sweep threshold from 0.2 to 0.8 (step 0.05). For each: approval rate, CoR, RoE, AUC-ROC, F1, recall (TPR), precision. Dual-axis chart: RoE and CoR vs threshold; optimal threshold marked.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

thresholds = np.arange(0.2, 0.85, 0.05)
auc_roc = roc_auc_score(y_test, p_test)
y_vals = np.asarray(y_test).ravel()
rows = []
for t in thresholds:
    approved_mask = (p_test < t)
    app = loans_df.iloc[approved_mask]
    approved_amnt = app["loan_amnt"].sum()
    charged_off_amnt = loans_df.iloc[approved_mask & (y_vals == 1)]["loan_amnt"].sum()
    n_approved = approved_mask.sum()
    approval_rate = n_approved / len(y_test)
    CoR = charged_off_amnt / approved_amnt if approved_amnt else 0
    wgt_int = (app["int_rate"] * app["loan_amnt"]).sum()
    wgt_int_rate = wgt_int / approved_amnt if approved_amnt else 0
    wgt_term = (app["term"] * app["loan_amnt"]).sum() / approved_amnt if approved_amnt else 0
    interest_earned = approved_amnt * (wgt_int_rate / 100) * (wgt_term / 12)
    interest_paid = approved_amnt * (COST_OF_FUNDS / 100) * (wgt_term / 12)
    banking_profit = interest_earned - interest_paid - charged_off_amnt
    RoE = (banking_profit / approved_amnt * 100) if approved_amnt else 0
    y_pred = (p_test >= t).astype(int)
    rows.append({
        "threshold": t,
        "approval_rate": approval_rate,
        "CoR": CoR,
        "RoE": RoE,
        "AUC_ROC": auc_roc,
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "precision": precision_score(y_test, y_pred, zero_division=0),
    })
sens_df = pd.DataFrame(rows)
print("Metrics by threshold:")
print(sens_df.round(4).to_string())
sens_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(sens_df["threshold"], sens_df["RoE"], color="tab:green", marker="o", markersize=4, label="RoE (%)")
ax1.set_xlabel("Threshold (approve if PD < threshold)")
ax1.set_ylabel("RoE (%)", color="tab:green")
ax1.tick_params(axis="y", labelcolor="tab:green")
ax1.axvline(x=best_threshold_opt, color="gray", linestyle="--", label=f"Optimal (F1) = {best_threshold_opt:.2f}")

ax2 = ax1.twinx()
ax2.plot(sens_df["threshold"], sens_df["CoR"] * 100, color="tab:red", marker="s", markersize=4, label="CoR (%)")
ax2.set_ylabel("CoR (%)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.title("RoE and Charge-off Rate vs Approval Threshold")
plt.tight_layout()
plt.show()

## 6. Baseline vs model – business case ($)

Compare $ approved (baseline vs model), $ charge-offs saved, and RoE improvement in percentage points.

In [ ]:
roe_baseline = (baseline_stats["approved_amnt"] * (baseline_stats["wgt_int_rate"]/100) * (baseline_stats["wgt_term"]/12)
roe_baseline = roe_baseline - baseline_stats["approved_amnt"] * (COST_OF_FUNDS/100) * (baseline_stats["wgt_term"]/12) - baseline_stats["charged_off_amnt"]
roe_baseline_pct = (roe_baseline / baseline_stats["approved_amnt"] * 100) if baseline_stats["approved_amnt"] else 0
roe_model_pct = (model_stats["approved_amnt"] * (model_stats["wgt_int_rate"]/100) * (model_stats["wgt_term"]/12)
roe_model_pct = roe_model_pct - model_stats["approved_amnt"] * (COST_OF_FUNDS/100) * (model_stats["wgt_term"]/12) - model_stats["charged_off_amnt"]
roe_model_pct = (roe_model_pct / model_stats["approved_amnt"] * 100) if model_stats["approved_amnt"] else 0

approved_baseline = baseline_stats["approved_amnt"]
approved_model = model_stats["approved_amnt"]
charged_off_baseline = baseline_stats["charged_off_amnt"]
charged_off_model = model_stats["charged_off_amnt"]
saved_charge_off = charged_off_baseline - charged_off_model
roe_improvement_pp = roe_model_pct - roe_baseline_pct

print("Business case (test cohort)")
print(f"  $ Loans approved – Baseline: ${approved_baseline/1e6:.2f}m  |  Model: ${approved_model/1e6:.2f}m")
print(f"  $ Charge-offs saved (vs baseline): ${saved_charge_off/1e6:.2f}m")
print(f"  RoE – Baseline: {roe_baseline_pct:.2f}%  |  Model: {roe_model_pct:.2f}%  |  Improvement: {roe_improvement_pp:+.2f} pp")

## 7. Summary

*Run the cell below to print a one-paragraph summary with the key numbers from this run.*

In [ ]:
summary = (
    f"At threshold {best_threshold_opt:.2f}, the model approves {model_stats['approval_rate']:.0%} of applications, "
    f"reducing CoR from {baseline_stats['CoR']:.1%} (baseline) to {model_stats['CoR']:.1%}, "
    f"improving portfolio RoE from {roe_baseline_pct:.1f}% to {roe_model_pct:.1f}%, "
    f"saving approximately ${saved_charge_off/1e6:.1f}m in charge-offs on the test cohort."
)
print(summary)